In [3]:
import random
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# CONFIGURATION
# =========================
NUM_SAMPLES_TOTAL = 40000            # total windows (2 steps each)
TRAIN_SIZE = 36000
TEST_SIZE = 2000
FINAL_TEST_SIZE = 2000

# Output paths
TRAIN_PATH = Path("../../../data/training")
TEST_PATH = Path("../../../data/test")
FINAL_TEST_PATH = Path("../../../data/final test")

for p in (TRAIN_PATH, TEST_PATH, FINAL_TEST_PATH):
    p.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ["Temperature", "Pressure", "RPM", "Vibration"]
WINDOW_LEN = 2  # 2-timestep context
CLASSES = ["Accelerating", "Steady", "Decelerating"]

# =========================
# HELPERS (relationships across full operating span)
# =========================
def gen_behaviour_window(state: str):

    # -------- RPM generation (UNCHANGED) --------
    rpm1 = random.uniform(1.0, 10000.0)

    if state == "Accelerating":
        delta = random.uniform(40.0, 300.0)
    elif state == "Decelerating":
        delta = -random.uniform(40.0, 300.0)
    else:  # Steady
        delta = random.uniform(-39.999999, 39.999999)

    rpm2 = max(0.0, rpm1 + delta)

    # -------- Independent auxiliary variables --------
    # Cover full project ranges so model never sees unseen values

    # Temperature full range
    temp1 = random.uniform(-100.0, 300.0)
    temp2 = random.uniform(-100.0, 300.0)

    # Pressure full range
    pres1 = random.uniform(-10.0, 10.0)
    pres2 = random.uniform(-10.0, 10.0)

    # Vibration full range
    vib1 = random.uniform(-5.0, 5.0)
    vib2 = random.uniform(-5.0, 5.0)

    window = [
        [temp1, pres1, rpm1, vib1],
        [temp2, pres2, rpm2, vib2],
    ]

    return window, state
# =========================
# GENERATE DATASET
# =========================

def generate_dataset(n_windows: int):
    """
    Balanced dataset across the three classes.
    CSV in long format:
    Time, Sequence, Temperature, Pressure, RPM, Vibration, State
    """
    X_list, y_list, rows = [], [], []
    per_class = n_windows // len(CLASSES)

    seq_counter = 1  # Sequence starts at 1

    for cls in CLASSES:
        for _ in range(per_class):
            window, label = gen_behaviour_window(cls)
            X_list.append(window)
            y_list.append(label)
            for t_idx, (temp, pres, rpm, vib) in enumerate(window, start=1):
                rows.append({
                    "Time": t_idx,
                    "Sequence": seq_counter,
                    "Temperature": temp,
                    "Pressure": pres,
                    "RPM": rpm,
                    "Vibration": vib,
                    "State": label
                })
            seq_counter += 1

    # If remainder, top up with random classes
    remaining = n_windows - per_class * len(CLASSES)
    for _ in range(remaining):
        cls = random.choice(CLASSES)
        window, label = gen_behaviour_window(cls)
        X_list.append(window)
        y_list.append(label)
        for t_idx, (temp, pres, rpm, vib) in enumerate(window, start=1):
            rows.append({
                "Time": t_idx,
                "Sequence": seq_counter,
                "Temperature": temp,
                "Pressure": pres,
                "RPM": rpm,
                "Vibration": vib,
                "State": label
            })
        seq_counter += 1

    X = np.array(X_list, dtype=np.float32)  # (N, 2, 4)
    y = np.array(y_list)                    # (N,)
    df = pd.DataFrame(rows, columns=["Time","Sequence","Temperature","Pressure","RPM","Vibration","State"])
    return X, y, df

# =========================
# BUILD SPLITS & SAVE
# =========================
# Training
X_train, y_train, df_train = generate_dataset(TRAIN_SIZE)
np.save(TRAIN_PATH / "Behaviour_training_X.npy", X_train)
np.save(TRAIN_PATH / "Behaviour_training_y.npy", y_train)
df_train.to_csv(TRAIN_PATH / "Behaviour_training.csv", index=False)

# Test
X_test, y_test, df_test = generate_dataset(TEST_SIZE)
np.save(TEST_PATH / "Behaviour_test_X.npy", X_test)
np.save(TEST_PATH / "Behaviour_test_y.npy", y_test)
df_test.to_csv(TEST_PATH / "Behaviour_test.csv", index=False)

# Final test
X_final, y_final, df_final = generate_dataset(FINAL_TEST_SIZE)
np.save(FINAL_TEST_PATH / "Behaviour_final test_X.npy", X_final)
np.save(FINAL_TEST_PATH / "Behaviour_final test_y.npy", y_final)
df_final.to_csv(FINAL_TEST_PATH / "Behaviour_final test.csv", index=False)

print("✅ Behaviour dataset created.")
print("Train X:", X_train.shape, "| Test X:", X_test.shape, "| Final X:", X_final.shape)


✅ Behaviour dataset created.
Train X: (36000, 2, 4) | Test X: (2000, 2, 4) | Final X: (2000, 2, 4)
